# Section 1: ShortVar Secondary Analysis Pipeline

Welcome to the **Short-Read Somatic Variant Secondary Analysis Pipeline** hands-on notebook. This notebook covers the end-to-end bioinformatics workflow for processing Next-Generation Sequencing (NGS) data, tailored to **Chromosome 17** (`chr17`).

### Pipeline Overview:
1. **Quality Control & Adapter Trimming** (`fastp`)
2. **Reference Genome Indexing** (`bwa index` & `samtools faidx`)
3. **Read Alignment & Read Group Tagging** (`bwa-mem` piped to `samtools sort`)
4. **PCR Duplicate Marking** (`picard MarkDuplicates`)
5. **Alignment Quality Statistics & Audit** (`samtools flagstat` / `samtools stats` + Pandas Table Import)
6. **Target Region Coverage & Depth Calculation** (`mosdepth` + Pandas Depth Tables & Summary)
7. **Somatic Variant Calling** (`GATK Mutect2`)
8. **Somatic Variant Filtering** (`GATK FilterMutectCalls` + Pandas Somatic VCF Table)


---  
## Step 1: FastP Quality Control & Trimming

### Theoretical Background

#### 1. Low Quality Reads & Phred Quality Scores
In sequencing by synthesis (Illumina), base calling quality is measured using **Phred quality scores** ($Q$). The Phred quality score is logarithmically related to the base calling error probability ($P$):

$$Q = -10 \log_{10}(P)$$

- **Q10**: Error probability $P = 10\% = 0.10$ (90% accuracy)
- **Q20**: Error probability $P = 1\% = 0.01$ (99% accuracy)
- **Q30**: Error probability $P = 0.1\% = 0.001$ (99.9% accuracy — standard threshold for high quality)

Accumulation of low-quality bases (due to phasing/pre-phasing, signal decay over cycles, or fluorophore degradation) leads to false positive alignment and variant calls.

#### 2. Adapter Contamination & Poly-G Tails
- **Adapter Read-Through**: Occurs when genomic insert lengths are shorter than the read length (e.g., insert size < 150 bp for 2x150 bp sequencing). The sequencer reads into synthetic library adapters (`AGATCGGAAGAG...`), which will fail to align or produce false soft-clipping if untrimmed.
- **Poly-G Tails**: Modern Illumina 2-color chemistry (e.g., NextSeq, NovaSeq) uses no-signal to denote Guanine ('G'). When optical intensity drops or at read ends, the instrument records artificial strings of 'GGGG...'. `fastp` automatically detects and trims poly-G tails.

> 💡 **Bioinformatics Interview Q&A: Quality Control & Phred Scores**
>
> **Q1: What is the Phred quality score ($Q30$) threshold and why is adapter read-through dangerous for variant calling?**
> **A:** A Phred score of Q30 represents a $1$ in $1,000$ base-calling error probability ($99.9\%$ base accuracy). Below Q30, accumulated base-call errors mimic low-frequency somatic mutations. Adapter read-through occurs when genomic fragment inserts are shorter than the sequencing read length. If adapters are untrimmed, the aligner will either fail to align the read or generate false soft-clipped bases, impairing downstream variant calling.
>
> **Q2: What causes artificial poly-G tails in modern Illumina sequencing?**
> **A:** Modern 2-color Illumina chemistry (e.g. NextSeq, NovaSeq) uses no optical signal (dark) to denote Guanine ('G'). When optical signal decays or at read ends, the instrument records artificial strings of 'GGGG...'. FastP detects and trims poly-G tails automatically.


In [ ]:
%%bash
mkdir -p Results/QC
fastp \
  -i Samples/SRR7890850_1.fastq.gz \
  -I Samples/SRR7890850_2.fastq.gz \
  -o Results/QC/trimmed_1.fastq.gz \
  -O Results/QC/trimmed_2.fastq.gz \
  -h Results/QC/fastp_report.html \
  -j Results/QC/fastp_report.json \
  --thread 4 \
  --detect_adapter_for_pe


---  
## Step 2: Indexing the Reference Genome

### Theoretical Background

#### Burrows-Wheeler Transform (BWT) & FM-Index
Aligning billions of short reads directly against a 3-billion-base reference genome line-by-line would require $O(N \times M)$ operations, which is computationally prohibitive. `bwa` uses the **Burrows-Wheeler Transform (BWT)** combined with an **FM-Index** (Full-text index in Minute space) and a **Suffix Array**.

1. **Burrows-Wheeler Transform (BWT)**: A reversible permutation of string characters that groups identical characters together, making the sequence highly compressible while enabling exact substring searches.
2. **LF Mapping Property**: The $i$-th occurrence of character $C$ in the Last column corresponds to the $i$-th occurrence of $C$ in the First column of the sorted rotations matrix.
3. **FM-Index**: Integrates BWT with prefix sums (Occurrence tables) to achieve string lookup in $O(m)$ time complexity (where $m$ is read length), completely independent of genome size $N$.

`bwa index` precomputes the `.bwt`, `.pac`, `.ann`, `.amb`, and `.sa` index files for chromosome 17, while `samtools faidx` creates `.fai` and `gatk CreateSequenceDictionary` creates `.dict` required by GATK.

> 💡 **Bioinformatics Interview Q&A: Genome Indexing & BWT**
>
> **Q: How does the FM-Index achieve $O(m)$ sequence alignment complexity, and why does GATK require a separate `.dict` sequence dictionary?**
> **A:** The FM-Index combines the Burrows-Wheeler Transform (BWT) with Suffix Array sampling and Last-First (LF) mapping count tables. It allows searching a query read of length $m$ in $O(m)$ time regardless of reference genome size $N$. GATK requires a SAM-format sequence dictionary (`.dict`) alongside the FASTA index (`.fai`) to validate chromosome names, sequence lengths, MD5 checksums, and assembly origin headers before processing BAM alignments.


In [ ]:
%%bash
mkdir -p References
bwa index References/chr17.fa
samtools faidx References/chr17.fa
gatk CreateSequenceDictionary -R References/chr17.fa -O References/chr17.dict


---  
## Step 3: Read Alignment & Sorting (`bwa-mem` + `samtools sort`)

### Theoretical Background

#### 1. Soft-Clipping vs Hard-Clipping
- **Soft-clipping (`S` tag in CIGAR string)**: Portion of the read at the 5' or 3' end that does not match the reference is excluded from alignment scoring but **retained in the BAM read sequence string**. Essential for detecting structural variations, breakpoints, and indels.
- **Hard-clipping (`H` tag in CIGAR string)**: Mismatched tail sequence is **permanently removed** from the BAM record to save space (commonly seen in supplementary alignments).

#### 2. `bwa-mem` vs Legacy `bwa-sampe` / `bwa-samse`
- **`bwa-backtrack` (`bwa aln` + `sampe`/`samse`)**: Designed for short reads (< 70 bp). Performs global alignment and relies on exact seed matches.
- **`bwa-mem`**: Uses Maximal Exact Matches (MEMs) to seed alignment, followed by Smith-Waterman local alignment extension. Handles longer reads (70 bp - 1 Mb), tolerates structural variation, indels, and performs soft-clipping naturally.

#### 3. Read Group (`@RG`) Metadata
GATK and downstream tools **require** valid Read Group tags to track batch effects, sequencing center, library, and lane information:
- `ID`: Unique Read Group Identifier (e.g., `SRR7890850`)
- `SM`: Sample Name (e.g., `TumorSample`)
- `PL`: Platform technology (e.g., `ILLUMINA`)
- `LB`: DNA preparation Library (e.g., `WES_Lib1`)
- `PU`: Platform Unit / Flowcell barcode

> 💡 **Bioinformatics Interview Q&A: Soft-Clipping & Read Groups**
>
> **Q1: What is the clinical significance of soft-clipping in CIGAR strings?**
> **A:** Soft-clipping (`S` in CIGAR) retains mismatched terminal read bases in the BAM file while masking them from alignment scoring. This preserved sequence is critical for split-read realignment algorithms to identify structural breakpoints, gene fusions, and large indels.
>
> **Q2: Why are Read Group (`@RG`) metadata tags strictly required by GATK?**
> **A:** Read Group (`@RG`) metadata tags (`ID`, `SM`, `PL`, `LB`, `PU`) are mandatory in GATK because tools like Mutect2 and BQSR rely on sample (`SM`) and library (`LB`) boundaries to model batch effects, optical duplicates, and platform-specific error rates.


In [ ]:
%%bash
mkdir -p Results/Align
bwa mem -t 4 \
  -R '@RG\tID:SRR7890850\tSM:TumorSample\tPL:ILLUMINA\tLB:WES_Lib1\tPU:Unit1' \
  References/chr17.fa \
  Results/QC/trimmed_1.fastq.gz \
  Results/QC/trimmed_2.fastq.gz | \
  samtools sort -@ 4 -o Results/Align/aligned_sorted.bam -

samtools index Results/Align/aligned_sorted.bam


---  
## Step 4: PCR Duplicate Marking (Picard MarkDuplicates)

### Theoretical Background

#### PCR Amplification Bias
During target enrichment library preparation (WES / Panel capture), PCR amplification can exponentially replicate certain DNA fragments disproportionately due to GC content bias or small insert size advantages.

- **PCR Duplicates**: Read pairs originating from the exact same physical DNA fragment during PCR amplification. They share identical 5' alignment coordinates and orientation.
- **Optical Duplicates**: Read clusters located physically adjacent on the flow cell surface due to optical sensor misreading single clusters as duplicates.

#### Impact on Variant Calling
If duplicates are not marked, a single PCR-amplified error can appear 50 times in coverage, creating a false somatic mutation with 100% variant allele frequency (VAF). `picard MarkDuplicates` tags duplicate reads with the SAM flag `0x400` (1024), allowing downstream variant callers like Mutect2 to ignore duplicate evidence.

> 💡 **Bioinformatics Interview Q&A: PCR Duplicate Marking**
>
> **Q: How does Picard identify PCR duplicates, and why are duplicates marked (flag `0x400`) rather than deleted in clinical pipelines?**
> **A:** Picard compares the 5' unclipped mapping coordinates and strand orientations of paired-end reads. Read pairs sharing identical 5' positions are flagged as PCR duplicates. Duplicates are marked with SAM flag `0x400` rather than deleted to maintain full data traceability for clinical audits and allow downstream callers or depth tools (like mosdepth) to decide whether to include or exclude duplicate reads.


In [ ]:
%%bash
mkdir -p Results/Deduplication
picard MarkDuplicates \
  I=Results/Align/aligned_sorted.bam \
  O=Results/Deduplication/dedup.bam \
  M=Results/Deduplication/marked_dup_metrics.txt \
  REMOVE_DUPLICATES=false

samtools index Results/Deduplication/dedup.bam


---  
## Step 5: Samtools BAM Statistics & Quality Audit

### Theoretical Background

#### Key Quality Audit Metrics
1. **Q30 Scores**: Proportion of base calls with Phred score $\ge 30$ ($> 99.9\%$ accuracy threshold).
2. **Percentage Mapped Reads**: Ratio of aligned reads to total sequenced reads ($> 95\%$ mapped is targeted threshold).
3. **Percentage Duplicates**: Ratio of marked duplicate reads relative to mapped reads ($< 15-20\%$ expected for good WES library complexity).
4. **Proper Pairs**: Reads where both forward and reverse reads align to the same chromosome within expected insert size orientation (FR orientation).

> 💡 **Bioinformatics Interview Q&A: GATK HaplotypeCaller/Mutect2 vs Naive Pileup Callers**
>
> **Q: How does GATK Mutect2/HaplotypeCaller differ fundamentally from naive pileup tools like Samtools mpileup?**
> **A:** Naive pileup callers (like Samtools mpileup) evaluate bases vertically coordinate-by-coordinate, trusting the initial BWA alignment completely. GATK Mutect2 and HaplotypeCaller perform **local de novo assembly**: when an active region with variation is detected, GATK discards the BWA alignment, constructs a De Bruijn graph to re-assemble the local reads into candidate haplotypes, and calculates Bayesian genotype likelihoods. This eliminates alignment artifacts around indels.


In [ ]:
%%bash
mkdir -p Results/Stats
samtools flagstat Results/Deduplication/dedup.bam > Results/Stats/bam_flagstat.txt
samtools stats Results/Deduplication/dedup.bam > Results/Stats/bam_stats.txt


### Inspect Alignment Statistics Table (Pandas)

Instead of plain shell text, we import the `samtools stats` Summary Numbers (`SN`) and `samtools flagstat` outputs into clean Pandas DataFrames.

In [ ]:
import pandas as pd
import os

# 1. Flagstat Data
flagstat_file = "Results/Stats/bam_flagstat.txt"
if os.path.exists(flagstat_file):
    with open(flagstat_file) as f:
        lines = [line.strip() for line in f if line.strip()]
    df_flagstat = pd.DataFrame({"Samtools Flagstat Metric": lines})
    print("=== Samtools Flagstat Summary ===")
    display(df_flagstat)

# 2. Detailed Summary Numbers (SN) from samtools stats
stats_file = "Results/Stats/bam_stats.txt"
if os.path.exists(stats_file):
    sn_records = []
    with open(stats_file) as f:
        for line in f:
            if line.startswith("SN\t"):
                parts = line.strip().split("\t")[1:]
                metric_name = parts[0].rstrip(":")
                metric_val = parts[1]
                sn_records.append({"Metric": metric_name, "Value": metric_val})
    df_sn = pd.DataFrame(sn_records)
    print("\n=== Samtools Stats Key Summary Numbers (SN) ===")
    display(df_sn.head(25))


---  
## Step 6: Mosdepth for Target Region Depth Analysis

### Theoretical Background

#### Efficient Depth Estimation
Calculating coverage over millions of target genomic windows using standard tools can be slow. `mosdepth` uses CIGAR string parsing with chromosome position arrays to achieve high-performance coverage calculation.

- **Target Region BED**: `Target_regions_chr17.bed` defines specific capture regions (exons/genes on chromosome 17).
- **Output Files**:
  - `.per-base.bed.gz`: Base-by-base depth.
  - `.regions.bed.gz`: Mean depth across each target region defined in the input BED file.
  - `.summary.txt`: Overall mean depth and fraction of bases covered at $> 1X, 10X, 20X, 50X, 100X$.

> 💡 **Bioinformatics Interview Q&A: Mosdepth vs Bedtools**
>
> **Q: Why is Mosdepth preferred over Bedtools coverage in clinical production pipelines?**
> **A:** Mosdepth utilizes CIGAR string parsing with fast C-array interval indexing rather than line-by-line streaming intersection. It is over $10\times$ faster than `bedtools coverage` and handles compressed BAM/CRAM files efficiently across multiple threads while outputting per-base and windowed summary coverage metrics.


In [ ]:
%%bash
mkdir -p Results/Coverage
mosdepth --by References/Target_regions_chr17.bed \
         --threads 4 \
         Results/Coverage/chr17_target_depth \
         Results/Deduplication/dedup.bam


### Inspect Target Depth Tables & Coverage Metrics (Pandas)

We load `mosdepth` summary statistics and per-target-region mean coverage using Pandas.

In [ ]:
import pandas as pd
import os

# 1. Mosdepth Depth Summary Table
summary_file = "Results/Coverage/chr17_target_depth.summary.txt"
if os.path.exists(summary_file):
    df_summary = pd.read_csv(summary_file, sep="\t")
    print("=== Mosdepth Target Region Coverage Summary ===")
    display(df_summary)

# 2. Target Regions Mean Depth Table
regions_file = "Results/Coverage/chr17_target_depth.regions.bed.gz"
if os.path.exists(regions_file):
    df_regions = pd.read_csv(
        regions_file, 
        sep="\t", 
        names=["Chromosome", "Start", "End", "Region_ID", "Mean_Depth"]
    )
    print("\n=== Target Regions Coverage Table (First 15 Regions) ===")
    display(df_regions.head(15))
    
    print("\n=== Target Regions Depth Distribution Stats ===")
    display(df_regions["Mean_Depth"].describe())


---  
## Step 7: Somatic Variant Calling using GATK Mutect2

### Theoretical Background

#### Bayesian Local Re-assembly Framework
`GATK Mutect2` uses **De Bruijn graph local re-assembly** over active genomic regions to detect single nucleotide variants (SNVs) and small insertions/deletions (indels).

#### Variant Calling Operational Modes:
1. **Tumor-Only Mode**: Calling variants on a tumor sample without matched normal tissue. Requires a **Panel of Normals (PoN)** and **Germline Population Resource** (e.g., gnomAD) to subtract common germline variants ($AF > 0.001$) and sequencing artifacts.
2. **Tumor-Normal Paired Mode**: Evaluates tumor vs matched normal BAM simultaneously. Provides highest sensitivity and specificity by subtracting patient-specific germline variations directly.
3. **Panel of Normals (PoN)**: Formed by running Mutect2 on multiple non-tumor samples to capture platform-dependent noise, repetitive element misalignments, and library preparation artifacts.

> 💡 **Bioinformatics Interview Q&A: Somatic vs Germline Calling & PoN**
>
> **Q: What is the primary difference between Somatic and Germline variant calling, and why is a Panel of Normals (PoN) critical in Tumor-Only mode?**
> **A:** Germline calling assumes a diploid state ($AF \approx 50\%$ or $100\%$). Somatic calling must detect low allele frequency subclonal mutations ($AF \le 5\%$) amid non-diploid tumor heterogeneity and normal tissue contamination. In Tumor-Only mode, a Panel of Normals (PoN) constructed from dozens of healthy samples is essential to subtract recurring sequencing noise, PCR artifacts, and false-positive mapping errors specific to the sequencing platform.


In [ ]:
%%bash
mkdir -p Results/Variants
gatk Mutect2 \
  -R References/chr17.fa \
  -I Results/Deduplication/dedup.bam \
  -L References/Target_regions_chr17.bed \
  -O Results/Variants/raw_somatic_variants.vcf.gz


---  
## Step 8: GATK Mutect2 Somatic Variant Filtering

### Theoretical Background

#### Filtering Artifacts & False Positives
Raw Mutect2 VCFs contain true somatic mutations along with technical artifacts and germline leakage. `FilterMutectCalls` applies statistical models to filter candidates:

- **Orientation Bias Filter (OxoG & FFPE)**:
  - **OxoG**: 8-oxoguanine oxidation during acoustic shearing converts G to 8-oxoG, causing artificial $G \rightarrow T$ / $C \rightarrow A$ mutations.
  - **FFPE**: Formalin-fixation deaminates Cytosine to Uracil, causing artificial $C \rightarrow T$ / $G \rightarrow A$ mutations.
- **Strand Bias Filter**: Detects variants supported predominantly by reads on only one strand (forward or reverse).
- **Clustered Events Filter**: Filters candidate variants clustered suspiciously close together in a short window.
- **`PASS` Filter Tag**: Variants passing all filter models are assigned `PASS` in column 7 of the VCF.

> 💡 **Bioinformatics Interview Q&A: OxoG, FFPE & Artifact Filtering**
>
> **Q: What are OxoG and FFPE orientation bias artifacts, and how does GATK FilterMutectCalls identify them?**
> **A:** OxoG artifacts arise during acoustic shearing when Guanine oxidizes to 8-oxoguanine, pairing with Adenine to create false $G \rightarrow T$ ($C \rightarrow A$) mutations. FFPE artifacts arise from formalin fixation deaminating Cytosine to Uracil, creating false $C \rightarrow T$ ($G \rightarrow A$) mutations. `FilterMutectCalls` uses strand-orientation bias modeling to verify whether candidate variants occur symmetrically on both forward and reverse read orientation pairs or exclusively on single-orientation strands.


In [ ]:
%%bash
gatk FilterMutectCalls \
  -R References/chr17.fa \
  -V Results/Variants/raw_somatic_variants.vcf.gz \
  -O Results/Variants/filtered_somatic_variants.vcf.gz


### Inspect Somatic Variants Table (Pandas & VCF Parsing)

We parse the filtered somatic VCF into a Pandas DataFrame to analyze called variants and filter statuses.

In [ ]:
import pandas as pd
import gzip
import os

vcf_file = "Results/Variants/filtered_somatic_variants.vcf.gz"
if os.path.exists(vcf_file):
    vcf_rows = []
    header_cols = []
    with gzip.open(vcf_file, "rt") as f:
        for line in f:
            if line.startswith("#CHROM"):
                header_cols = line.strip("#\n").split("\t")
            elif not line.startswith("#"):
                vcf_rows.append(line.strip().split("\t"))
                
    if vcf_rows and header_cols:
        df_vcf = pd.DataFrame(vcf_rows, columns=header_cols)
        print(f"Total Somatic Candidate Variants Called: {len(df_vcf)}")
        print("\n=== Filter Status Breakdown ===")
        display(df_vcf["FILTER"].value_counts().to_frame())
        
        print("\n=== Somatic Variants Table (First 20 Candidates) ===")
        main_cols = [c for c in ["CHROM", "POS", "REF", "ALT", "FILTER", "TumorSample"] if c in df_vcf.columns]
        display(df_vcf[main_cols].head(20))
    else:
        print("No somatic variants recorded in the VCF file.")
